In [1]:
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, \
    HistGradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline

from sklearn.model_selection import cross_validate
from features.preprocess import build_preprocessor
from utils.helper import get_data_path
import pandas as pd
import numpy as np

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

RANDOM_STATE = 42

In [2]:
df = pd.read_csv(get_data_path("final_data.csv"))
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

X_train = train_df.drop("price", axis=1)
y_train = np.log1p(train_df["price"] * 1000 / train_df["area"])

X_test = test_df.drop("price", axis=1)
y_test = np.log1p(test_df["price"] * 1000 / test_df["area"])
print(f"\nTrain size: {len(X_train)}, Test size: {len(X_test)}")
print(f"Features: {X_train.shape[1]} columns")


Train size: 16274, Test size: 4069
Features: 14 columns


In [13]:
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(random_state=RANDOM_STATE),
    "Lasso": Lasso(random_state=RANDOM_STATE, max_iter=10000),
    "ElasticNet": ElasticNet(random_state=RANDOM_STATE),

    "DecisionTree": DecisionTreeRegressor(random_state=RANDOM_STATE),
    "RandomForest": RandomForestRegressor(random_state=RANDOM_STATE),

    "GradientBoosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
    "HistGB": HistGradientBoostingRegressor(random_state=RANDOM_STATE),

    "XGBoost": XGBRegressor(random_state=RANDOM_STATE, verbosity=0),
    "LightGBM": LGBMRegressor(random_state=RANDOM_STATE, verbose=-1),
    "CatBoost": CatBoostRegressor(random_state=RANDOM_STATE, verbose=0)
}

In [8]:
k = 3
cv = KFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE)

In [5]:
scoring = {
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2"
}

In [10]:

numerical_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)

# numerical features - preprocessing steps
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]
)

numerical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

Numerical features: ['area', 'floors', 'bedrooms', 'bathrooms', 'year', 'lat', 'lng']
Categorical features: ['address', 'house_direction', 'balcony_direction', 'legal_status', 'furniture_state', 'property_type', 'property_feature']


In [14]:
rows = []

for name, model in models.items():
    pipe = Pipeline([
        ("prep", preprocess),
        ("model", model)
    ])
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    rows.append({
        "model": name,
        "cv_rmse": -scores["test_rmse"].mean(), 
        "cv_mae": -scores["test_mae"].mean(),
        "cv_r2": scores["test_r2"].mean()
    })

# sort based on lowest rmse value
cv_results = pd.DataFrame(rows).sort_values("cv_rmse")
print("=== CV Model Comparison ===")
print(cv_results)

=== CV Model Comparison ===
               model   cv_rmse    cv_mae     cv_r2
9           LightGBM  0.213007  0.155873  0.764553
7             HistGB  0.214754  0.157021  0.760668
5       RandomForest  0.215669  0.151872  0.758601
10          CatBoost  0.216110  0.158763  0.757647
8            XGBoost  0.219512  0.160247  0.749953
6   GradientBoosting  0.229269  0.169276  0.727238
4       DecisionTree  0.281051  0.194016  0.590104
1              Ridge  0.303128  0.202796  0.520977
0   LinearRegression  0.327568  0.196628  0.435986
3         ElasticNet  0.438957  0.332697 -0.000043
2              Lasso  0.438957  0.332697 -0.000043


In [15]:
best_row = cv_results.sort_values("cv_rmse").iloc[0]

best_model_name = best_row["model"]
best_rmse = best_row["cv_rmse"]

print("Best model based on CV RMSE:")
print("Model :", best_model_name)
print("CV RMSE:", best_rmse)

Best model based on CV RMSE:
Model : LightGBM
CV RMSE: 0.21300738450997403
